In [11]:
import numpy as np
import pandas as pd
import glob
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

In [12]:


RES_GLOB = "louvain_result_res*.csv"  


LABELS_CSV = "~/stellar-clustering/publication/labeled-data/normalization/labels_mapped_normalized.csv"      


OUT_CSV = "louvain_eval_by_resolution.csv"

In [13]:
labels_df = pd.read_csv(LABELS_CSV)

In [14]:
def compute_purity(true_labels, pred_labels):
    conf_matrix = confusion_matrix(true_labels, pred_labels)
    purity = np.sum(np.max(conf_matrix, axis=0)) / np.sum(conf_matrix)
    return purity

In [15]:
result_files = sorted(glob.glob(RES_GLOB))

In [ ]:
rows = []

for file_path in result_files:
    
    louvain_df = pd.read_csv(file_path)
    
    resolution = louvain_df['resolution'].iloc[0]
    
    merged_df = louvain_df.merge(labels_df, on='account_id', how='inner')
    
    
    le = LabelEncoder()
    true_labels = le.fit_transform(merged_df['name'].values)
    pred_labels = merged_df['community'].values
    
    rows.append({
        'file': file_path,
        'resolution': resolution,
        'n_samples': len(true_labels),
        'true_clusters': len(np.unique(true_labels)),
        'pred_clusters': len(np.unique(pred_labels)),
        'nmi': normalized_mutual_info_score(true_labels, pred_labels),
        'ari': adjusted_rand_score(true_labels, pred_labels),
        'purity': compute_purity(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'recall': recall_score(true_labels, pred_labels, average='weighted', zero_division=0),
        'f1': f1_score(true_labels, pred_labels, average='weighted', zero_division=0)
    })

In [17]:
df_results = pd.DataFrame(rows).sort_values('resolution')
df_results.to_csv(OUT_CSV, index=False)
display(df_results)

,file,resolution,n_samples,true_clusters,pred_clusters,nmi,ari,purity,precision,recall,f1
0,louvain_result_res0.5.csv,0.5,8336,212,17,0.119071,0.045438,0.935221,0.0,0.0,0.0
1,louvain_result_res0.8.csv,0.8,8336,212,20,0.117005,0.045747,0.934021,0.0,0.0,0.0
2,louvain_result_res1.0.csv,1.0,8336,212,21,0.117704,0.044763,0.934381,0.0,0.0,0.0
3,louvain_result_res1.2.csv,1.2,8336,212,23,0.116123,0.043489,0.934141,0.0,0.0,0.0
